# CUAD ModernBERT experiment
Adapted from previous Version 4 preprocessing and BERT notebooks. Preserves contract splits and per-category label masking. Validation selects thresholds; test remains untouched. Start with a pilot; pilot metrics are not full-training results.

In [1]:
%pip install -q "transformers==4.57.6" scikit-learn

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
from google.colab import drive, files
from pathlib import Path
import torch, json, random, numpy as np
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU"
print(torch.cuda.get_device_name(0))
drive.mount('/content/drive')
BASE_DIR = Path('/content/cuad_inputs')
BASE_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = Path('/content/drive/MyDrive/CUAD_ModernBERT')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
required = ['train_separate_questions.json', 'test.json', 'cuad_categories.json']
if not all((BASE_DIR / f).exists() for f in required):
    uploaded = files.upload()
    for name in required:
        if name in uploaded:
            (BASE_DIR / name).write_bytes(uploaded[name])
assert all((BASE_DIR / f).exists() for f in required), 'Upload all three input files'


Tesla T4
Mounted at /content/drive


Saving train_separate_questions.json to train_separate_questions.json
Saving test.json to test.json
Saving cuad_categories.json to cuad_categories.json


In [3]:
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification
MODEL_NAME = 'answerdotai/ModernBERT-base'
MAX_LENGTH = 1024
STRIDE = 256
RANDOM_STATE = 42
PROCESSED_DIR = BASE_DIR
PILOT = True # False uses all training windows
PILOT_WINDOWS = 2000
TRAIN_EPOCHS = 1
BATCH_SIZE = 2
ACCUMULATION = 4
LEARNING_RATE = 2e-5
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
assert tokenizer.is_fast


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

In [4]:
def read_contracts(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing dataset: {path}. Check the notebook working directory.")
    return json.loads(path.read_text(encoding="utf-8"))["data"]

source_train_contracts = read_contracts(BASE_DIR / "train_separate_questions.json")
test_contracts = read_contracts(BASE_DIR / "test.json")
categories = json.loads((PROCESSED_DIR / "cuad_categories.json").read_text(encoding="utf-8"))

assert len(source_train_contracts) == 408 and len(test_contracts) == 102, "Unexpected official data split"
assert len(categories) == 41 and len(set(categories)) == 41, "Expected 41 unique CUAD categories"

train_contracts, validation_contracts = train_test_split(
    source_train_contracts, test_size=0.2, random_state=RANDOM_STATE, shuffle=True
)
splits = {
    "train": train_contracts,
    "validation": validation_contracts,
    "test": test_contracts,
}

title_sets = {name: {c["title"] for c in items} for name, items in splits.items()}
assert all(len(title_sets[name]) == len(items) for name, items in splits.items()), "Duplicate titles in split"
assert not (title_sets["train"] & title_sets["validation"])
assert not (title_sets["train"] & title_sets["test"])
assert not (title_sets["validation"] & title_sets["test"])
assert (len(train_contracts), len(validation_contracts), len(test_contracts)) == (326,82,102)

label2id = {name: i for i, name in enumerate(categories)}
id2label = {str(i): name for name, i in label2id.items()}
print("Contract splits:", {name: len(items) for name, items in splits.items()})
print("Labels:", len(label2id))

Contract splits: {'train': 326, 'validation': 82, 'test': 102}
Labels: 41


In [5]:
def contract_text(contract):
    return " ".join(p["context"] for p in contract["paragraphs"])

def get_answer_spans(contract):
    spans = []
    offset = 0
    for paragraph in contract["paragraphs"]:
        context = paragraph["context"]
        for qa in paragraph["qas"]:
            if qa.get("is_impossible", False):
                continue
            for answer in qa["answers"]:
                start = offset + answer["answer_start"]
                end = start + len(answer["text"])
                spans.append((start, end, qa["id"], answer["text"]))
        offset += len(context) + 1
    return spans

def category_from_id(qid):
    category = qid.rsplit("__", 1)[-1]
    suffix = category.rsplit("_", 1)
    return suffix[0] if len(suffix) == 2 and suffix[1].isdigit() else category

invalid = []
unknown_categories = []
for split_name, items in splits.items():
    for c in items:
        text = contract_text(c)
        for start, end, qid, expected_answer in get_answer_spans(c):
            if text[start:end] != expected_answer:
                invalid.append((split_name, c["title"], qid))
            if category_from_id(qid) not in label2id:
                unknown_categories.append((split_name, qid, category_from_id(qid)))
print("Invalid annotation offsets:", len(invalid))
print("Unknown label categories:", len(unknown_categories))
assert not invalid, f"Fix offset mismatch first: {invalid[:3]}"
assert not unknown_categories, f"Check label mapping first: {unknown_categories[:3]}"

Invalid annotation offsets: 0
Unknown label categories: 0


In [6]:
def windows_for_contract(contract, split_name):
    text = contract_text(contract)
    spans = get_answer_spans(contract)
    if not text.strip():
        return [], 0, []

    encoded = tokenizer(
        text,
        max_length=MAX_LENGTH,
        truncation=True,
        stride=STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        add_special_tokens=True,
        padding=False,
    )

    rows = []
    inference_rows = []
    windows_with_partials = 0

    for win_i, (ids, attention_mask, offsets) in enumerate(zip(
        encoded["input_ids"], encoded["attention_mask"], encoded["offset_mapping"]
    )):
        assert len(ids) <= MAX_LENGTH, "ModernBERT length limit violated"
        content_offsets = [(s, e) for s, e in offsets if e > s]
        if not content_offsets:
            continue

        begin = min(s for s, e in content_offsets)
        finish = max(e for s, e in content_offsets)

        # These are the SAME fixed windows we will use on unseen contracts.
        inference_row = {
            "contract_title": contract["title"],
            "window_id": win_i,
            "char_start": begin,
            "char_end": finish,
            "input_ids": ids,
            "attention_mask": attention_mask,
        }
        if "token_type_ids" in encoded:
            inference_row["token_type_ids"] = encoded["token_type_ids"][win_i]
        inference_rows.append(inference_row)

        full_indices = []
        full_categories = set()
        partial_categories = set()

        for span_i, (a, b, qid, answer_text) in enumerate(spans):
            if a < finish and begin < b:  # Answer overlaps this window.
                category = category_from_id(qid)
                if begin <= a and b <= finish:
                    full_indices.append(span_i)
                    full_categories.add(category)
                else:
                    partial_categories.add(category)

        if partial_categories:
            windows_with_partials += 1

        labels = [int(category in full_categories) for category in categories]

        # A full answer makes a positive label usable even if another
        # answer of the same category partly overlaps the window.
        label_mask = [
            int(category in full_categories or category not in partial_categories)
            for category in categories
        ]

        row = {
            "split": split_name,
            "contract_title": contract["title"],
            "window_id": win_i,
            "char_start": begin,
            "char_end": finish,
            "text": text[begin:finish],
            "input_ids": ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "label_mask": label_mask,
            "label_names": sorted(full_categories),
            "partial_label_names": sorted(partial_categories - full_categories),
            # For coverage auditing; the model does not use these fields.
            "covered_span_indices": full_indices,
        }
        if "token_type_ids" in encoded:
            row["token_type_ids"] = encoded["token_type_ids"][win_i]
        rows.append(row)

    assert len(rows) == len(inference_rows), "Labeled and inference windows should match"
    return rows, windows_with_partials, inference_rows


all_rows = {}
all_inference_rows = {}
partial_window_counts = {}

for name, items in splits.items():
    all_rows[name] = []
    all_inference_rows[name] = []
    partial_window_counts[name] = 0
    for contract in items:
        rows, partial_count, inference_rows = windows_for_contract(contract, name)
        all_rows[name].extend(rows)
        all_inference_rows[name].extend(inference_rows)
        partial_window_counts[name] += partial_count

    print(
        name,
        "saved windows:", len(all_rows[name]),
        "windows with at least one partial answer:", partial_window_counts[name],
        "ignored label positions:", sum(
            row["label_mask"].count(0) for row in all_rows[name]
        ),
    )

train saved windows: 5211 windows with at least one partial answer: 993 ignored label positions: 737
validation saved windows: 1042 windows with at least one partial answer: 174 ignored label positions: 133
test saved windows: 1365 windows with at least one partial answer: 258 ignored label positions: 185


In [7]:
coverage_report = {}
missing_answers = []

for name, items in splits.items():
    rows_by_title = {}
    for row in all_rows[name]:
        rows_by_title.setdefault(row["contract_title"], []).append(row)

    total = covered = too_long = 0
    for contract in items:
        title = contract["title"]
        for span_i, (start, end, qid, answer) in enumerate(get_answer_spans(contract)):
            total += 1
            category = category_from_id(qid)
            category_i = label2id[category]

            found = any(
                span_i in row["covered_span_indices"]
                and row["labels"][category_i] == 1
                and row["label_mask"][category_i] == 1
                for row in rows_by_title.get(title, [])
            )
            if found:
                covered += 1
                continue

            # Count subwords without trying to pass an overlength input into BERT.
            token_count = len(tokenizer.tokenize(answer)) + tokenizer.num_special_tokens_to_add(pair=False)
            if token_count > MAX_LENGTH:
                too_long += 1
            missing_answers.append({
                "split": name,
                "contract_title": title,
                "qa_id": qid,
                "answer_start": start,
                "answer_end": end,
                "answer_tokens_with_specials": token_count,
                "reason": "answer_over_limit" if token_count > MAX_LENGTH else "not_fully_contained_in_any_window",
            })

    rows = all_rows[name]
    coverage_report[name] = {
        "contracts": len(items),
        "annotated_answers": total,
        "answers_covered_by_windows": covered,
        "coverage_percent": round(100 * covered / total, 2) if total else None,
        "missing_answers": total - covered,
        "missing_because_answer_too_long": too_long,
        "windows_with_partial_annotations": partial_window_counts[name],
        "windows_excluded_due_to_partial": 0,
        "saved_windows": len(rows),
        "masked_label_positions": sum(row["label_mask"].count(0) for row in rows),
        "all_inference_windows": len(all_inference_rows[name]),
        "positive_windows": sum(any(row["labels"]) for row in rows),
        "no_positive_labels_windows": sum(not any(row["labels"]) for row in rows),
        "max_input_tokens": max((len(row["input_ids"]) for row in rows), default=0),
    }
    print(name, coverage_report[name])

assert all(len(row["input_ids"]) <= MAX_LENGTH
           for rows in all_rows.values() for row in rows)
assert all(len(row["labels"]) == len(categories)
           and len(row["label_mask"]) == len(categories)
           for rows in all_rows.values() for row in rows)
assert all(len(row["attention_mask"]) == len(row["input_ids"])
           for rows in all_rows.values() for row in rows)
assert all(
    all(label == 0 or mask == 1 for label, mask in zip(row["labels"], row["label_mask"]))
    for rows in all_rows.values() for row in rows
), "A positive label must never be ignored"
assert all(len(all_rows[name]) == len(all_inference_rows[name])
           for name in splits), "Every fixed window should be retained"
assert all_rows["train"] and all_rows["validation"] and all_rows["test"]
print("\nChecks passed: token lengths, 41 labels, masks, window counts, nonempty splits")
print("Uncovered annotation examples:", missing_answers[:3])

train {'contracts': 326, 'annotated_answers': 9175, 'answers_covered_by_windows': 9147, 'coverage_percent': 99.69, 'missing_answers': 28, 'missing_because_answer_too_long': 0, 'windows_with_partial_annotations': 993, 'windows_excluded_due_to_partial': 0, 'saved_windows': 5211, 'masked_label_positions': 737, 'all_inference_windows': 5211, 'positive_windows': 3028, 'no_positive_labels_windows': 2183, 'max_input_tokens': 1024}
validation {'contracts': 82, 'annotated_answers': 2005, 'answers_covered_by_windows': 2004, 'coverage_percent': 99.95, 'missing_answers': 1, 'missing_because_answer_too_long': 0, 'windows_with_partial_annotations': 174, 'windows_excluded_due_to_partial': 0, 'saved_windows': 1042, 'masked_label_positions': 133, 'all_inference_windows': 1042, 'positive_windows': 610, 'no_positive_labels_windows': 432, 'max_input_tokens': 1024}
test {'contracts': 102, 'annotated_answers': 2643, 'answers_covered_by_windows': 2640, 'coverage_percent': 99.89, 'missing_answers': 3, 'missin

In [8]:
from torch.utils.data import DataLoader
import torch.nn.functional as F
from sklearn.metrics import f1_score, precision_score, recall_score
from tqdm.auto import tqdm
train_rows = all_rows['train']
selected_train = random.Random(42).sample(train_rows, min(PILOT_WINDOWS, len(train_rows))) if PILOT else train_rows
val_rows = all_inference_rows['validation']
val_gold = [{'contract_title': x['title'], 'labels': [int(k in {category_from_id(qid) for _,_,qid,_ in get_answer_spans(x)}) for k in categories]} for x in validation_contracts]
def collate(batch):
    inputs = tokenizer.pad([{'input_ids': r['input_ids'], 'attention_mask': r['attention_mask']} for r in batch], padding=True, pad_to_multiple_of=8, return_tensors='pt')
    if 'labels' in batch[0]:
        inputs['labels'] = torch.tensor([r['labels'] for r in batch], dtype=torch.float32)
        inputs['label_mask'] = torch.tensor([r['label_mask'] for r in batch], dtype=torch.float32)
    return inputs
train_loader = DataLoader(selected_train, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader = DataLoader(val_rows, batch_size=BATCH_SIZE, collate_fn=collate)
y = np.array([r['labels'] for r in selected_train])
m = np.array([r['label_mask'] for r in selected_train])
pos = (y*m).sum(0)
neg = ((1-y)*m).sum(0)
weights = np.where(pos>0, np.clip(np.sqrt(neg/np.maximum(pos,1)),1,10),1)
print('Training windows:', len(selected_train), 'Validation windows:', len(val_rows))
print('Positive weights:', weights)


Training windows: 2000 Validation windows: 1042
Positive weights: [ 8.53966779 10.          3.87298335  3.53697669  3.97875715  3.81666466
  6.99106573  7.96565207  7.07469307  3.60361124  4.14290193  4.81613143
  3.4745314   3.52308986  5.05568986  4.981785    7.3711148   8.88144132
  3.69283591  8.54183604  4.41412914 10.         10.          9.69290364
  6.10800232  9.70027     6.04542553  6.2401923   3.4612887   4.53608719
 10.          5.16801058  4.59830887  6.43791958 10.          5.83095189
 10.          6.50919083 10.          7.59837444  7.8322251 ]


In [9]:
device = torch.device('cuda')
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=41, problem_type='multi_label_classification', id2label={i:k for k,i in label2id.items()}, label2id=label2id, attn_implementation='sdpa')
model.to(device)
model.gradient_checkpointing_enable()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scaler = torch.amp.GradScaler('cuda')
positive_weights = torch.tensor(weights, dtype=torch.float32, device=device)
for epoch in range(TRAIN_EPOCHS):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    progress = tqdm(train_loader, desc=f'Epoch {epoch+1}')
    for step, batch in enumerate(progress, 1):
        labels = batch.pop('labels').to(device)
        mask = batch.pop('label_mask').to(device)
        batch = {k:v.to(device) for k,v in batch.items()}
        with torch.autocast('cuda', dtype=torch.float16):
            logits = model(**batch).logits
            losses = F.binary_cross_entropy_with_logits(logits.float(), labels, pos_weight=positive_weights, reduction='none')
            loss = (losses*mask).sum()/mask.sum().clamp_min(1)
        group_size = min(ACCUMULATION, len(train_loader)-((step-1)//ACCUMULATION)*ACCUMULATION)
        scaler.scale(loss/group_size).backward()
        if step % ACCUMULATION == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
        progress.set_postfix(loss=round(loss.item(),4))
    checkpoint = OUTPUT_DIR / ('pilot' if PILOT else 'full')
    model.save_pretrained(checkpoint)
    tokenizer.save_pretrained(checkpoint)
print('Checkpoint saved:', checkpoint)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1:   0%|          | 0/1000 [00:00<?, ?it/s]

You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
W1119 01:05:38.758000 639 torch/_inductor/utils.py:1806] [1/0_1] Not enough SMs to use max_autotune_gemm mode


Checkpoint saved: /content/drive/MyDrive/CUAD_ModernBERT/pilot


In [10]:
model.eval()
by_contract = {}
with torch.inference_mode():
    for i,batch in enumerate(tqdm(val_loader, desc='Validation')):
        with torch.autocast('cuda', dtype=torch.float16):
            probs = torch.sigmoid(model(**{k:v.to(device) for k,v in batch.items()}).logits.float()).cpu().numpy()
        for row, prob in zip(val_rows[i*BATCH_SIZE:(i+1)*BATCH_SIZE], probs):
            title = row['contract_title']
            by_contract[title] = np.maximum(by_contract.get(title, np.zeros(41)),prob)
y_true = np.array([r['labels'] for r in val_gold])
y_prob = np.stack([by_contract[r['contract_title']] for r in val_gold])
def metrics(t):
    pred = y_prob >= t
    return dict(threshold=float(t), macro_f1=float(f1_score(y_true,pred,average='macro',zero_division=0)), micro_f1=float(f1_score(y_true,pred,average='micro',zero_division=0)), precision_micro=float(precision_score(y_true,pred,average='micro',zero_division=0)), recall_micro=float(recall_score(y_true,pred,average='micro',zero_division=0)))
candidates = [metrics(t) for t in np.arange(.1,.91,.05)]
best = max(candidates,key=lambda x:x['macro_f1'])
report = dict(model=MODEL_NAME, pilot=PILOT, training_windows=len(selected_train), validation_contracts=len(val_gold), max_length=MAX_LENGTH, overlap=STRIDE, epochs=TRAIN_EPOCHS, batch_size=BATCH_SIZE, accumulation=ACCUMULATION, learning_rate=LEARNING_RATE, positive_weights=weights.tolist(), label2id=label2id, coverage=coverage_report, validation_metrics=best, threshold_candidates=candidates, fixed_threshold_metrics=metrics(.5), split_titles={k:sorted(v) for k,v in title_sets.items()})
(checkpoint/'training_report.json').write_text(json.dumps(report,indent=2))
np.savez_compressed(checkpoint/'validation_predictions.npz',y_true=y_true,y_prob=y_prob,contract_titles=[r['contract_title'] for r in val_gold])
print(json.dumps(best,indent=2))
print('PILOT ONLY' if PILOT else 'FULL TRAINING VALIDATION', '| Saved:', checkpoint)


Validation:   0%|          | 0/521 [00:00<?, ?it/s]

{
  "threshold": 0.1,
  "macro_f1": 0.428360378659144,
  "micro_f1": 0.5475138121546961,
  "precision_micro": 0.38277327153341056,
  "recall_micro": 0.9612027158098934
}
PILOT ONLY | Saved: /content/drive/MyDrive/CUAD_ModernBERT/pilot


## Next steps
Compare against BERT on identical validation titles and threshold-selection criteria. Pilot results use fewer training windows. For a full run set PILOT=False and rerun from settings onward. Test evaluation is reserved for after model selection. This experiment changes several factors at once, so improvements cannot be attributed to architecture alone.